# 04 - Modelo LDA

**Grupo 15 | UMSS FCyT | IA I/2026**

Objetivos:
- Crear diccionario y corpus en formato gensim
- Buscar el número óptimo de tópicos K
- Entrenar el modelo LDA final
- Explorar e interpretar los tópicos descubiertos
- Guardar el modelo entrenado

In [ ]:
import sys, pickle, warnings
sys.path.append('../')
warnings.filterwarnings('ignore')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

from src.preprocesamiento import procesar_corpus
from src.modelo import (
    crear_diccionario_y_corpus,
    entrenar_lda,
    calcular_coherencia,
    optimizar_num_topics,
    obtener_topicos,
    asignar_topicos,
    parsear_topico_palabras,
)
from src.visualizacion import grafico_coherencia_vs_topicos
print('Módulos cargados ✅')

## 1. Cargar y preprocesar corpus

In [ ]:
df = pd.read_csv('../data/raw/encuesta_sintetica.csv')
textos = df['comentario'].astype(str).tolist()

print(f'Documentos: {len(textos)}')
print(f'Ejemplo original: {textos[0]}')

# Preprocesar (tokenizar + stopwords + stemming)
corpus_tokens = procesar_corpus(textos)
# Filtrar tokens con longitud < 3
corpus_tokens = [[t for t in doc if len(t) >= 3] for doc in corpus_tokens]

print(f'\nEjemplo procesado: {corpus_tokens[0]}')
print(f'Docs no vacíos: {sum(1 for d in corpus_tokens if len(d) > 0)}')

## 2. Crear diccionario y corpus gensim

In [ ]:
dictionary, corpus_gensim = crear_diccionario_y_corpus(corpus_tokens)

print(f'Tamaño del diccionario : {len(dictionary)} términos')
print(f'Documentos en corpus   : {len(corpus_gensim)}')
print(f'\nPrimer doc (BoW, 5 primeros términos):')
for tid, freq in corpus_gensim[0][:5]:
    print(f'  [{tid}] "{dictionary[tid]}" → freq={freq}')

## 3. Buscar K óptimo (coherencia c_v)

In [ ]:
print('Buscando K óptimo (K=2..10)...')
resultados_coherencia = optimizar_num_topics(
    corpus_gensim, dictionary, corpus_tokens,
    min_topics=2, max_topics=10, step=1, passes=10
)

num_optimo = max(resultados_coherencia, key=resultados_coherencia.get)
print(f'\n✅ K óptimo: {num_optimo} (coherencia={resultados_coherencia[num_optimo]:.4f})')

In [ ]:
fig, ax = grafico_coherencia_vs_topicos(resultados_coherencia)
plt.savefig('../data/processed/fig_coherencia_vs_k.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\nCoherencia por K:')
for k, c in sorted(resultados_coherencia.items()):
    marker = ' ← ÓPTIMO' if k == num_optimo else ''
    print(f'  K={k:2d} → {c:.4f}{marker}')

## 4. Entrenar modelo LDA final

In [ ]:
print(f'Entrenando LDA con K={num_optimo}, passes=20...')
lda_model = entrenar_lda(
    corpus_gensim,
    dictionary,
    num_topics=num_optimo,
    passes=20,
)

coherencia_final = calcular_coherencia(lda_model, corpus_gensim, dictionary, corpus_tokens)
print(f'✅ Modelo entrenado | Coherencia final: {coherencia_final:.4f}')

## 5. Explorar tópicos descubiertos

In [ ]:
topicos = obtener_topicos(lda_model, num_words=12)

print('=' * 55)
print(f'TÓPICOS DESCUBIERTOS (K={num_optimo})')
print('=' * 55)
for nombre, palabras_str in topicos.items():
    print(f'\n🏷️  {nombre}')
    palabras_dict = parsear_topico_palabras(palabras_str)
    top_words = sorted(palabras_dict.items(), key=lambda x: x[1], reverse=True)[:8]
    for w, p in top_words:
        bar = '█' * int(p * 500)
        print(f'   {w:<20} {bar} ({p:.4f})')

In [ ]:
from wordcloud import WordCloud

n_cols = 3
n_rows = -(-num_optimo // n_cols)  # techo
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 5, n_rows * 3.5))
axes = axes.flatten()

for i, (nombre, palabras_str) in enumerate(topicos.items()):
    palabras_dict = parsear_topico_palabras(palabras_str)
    if palabras_dict:
        wc = WordCloud(width=400, height=250,
                       background_color='white', colormap='viridis'
                       ).generate_from_frequencies(palabras_dict)
        axes[i].imshow(wc, interpolation='bilinear')
        axes[i].axis('off')
        axes[i].set_title(nombre, fontweight='bold')

for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.suptitle('Nubes de palabras por tópico', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/fig_wordclouds.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Asignar tópicos a documentos

In [ ]:
topicos_doc = asignar_topicos(lda_model, corpus_gensim)

df['topico_dominante'] = [t[0] if t[0] is not None else -1 for t in topicos_doc]
df['prob_topico']      = [t[1] for t in topicos_doc]

print('Distribución de tópicos dominantes:')
print(df['topico_dominante'].value_counts().sort_index())

fig, ax = plt.subplots(figsize=(8, 3))
counts = df['topico_dominante'].value_counts().sort_index()
ax.bar([f'T{i}' for i in counts.index], counts.values, color='coral')
ax.set_title('Documentos por tópico dominante', fontweight='bold')
ax.set_ylabel('Cantidad')
plt.tight_layout()
plt.savefig('../data/processed/fig_dist_topicos.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Guardar modelo y metadatos

In [ ]:
import os
os.makedirs('../data/processed', exist_ok=True)

# Guardar modelo gensim
lda_model.save('../data/processed/modelo_lda.model')
dictionary.save('../data/processed/diccionario.dict')

# Guardar corpus tokenizado
with open('../data/processed/corpus_procesado.pkl', 'wb') as f:
    pickle.dump(corpus_tokens, f)

# Guardar DataFrame con tópicos asignados
df.to_csv('../data/processed/encuesta_con_topicos.csv', index=False)

# Guardar metadatos
metadatos = {
    'num_topics': num_optimo,
    'num_documents': len(corpus_gensim),
    'vocab_size': len(dictionary),
    'coherencia_cv': coherencia_final,
    'passes': 20,
}
with open('../data/processed/metadatos_modelo.pkl', 'wb') as f:
    pickle.dump(metadatos, f)

print('✅ Modelo guardado en data/processed/')
print(f'   K={num_optimo} | Coherencia={coherencia_final:.4f} | Vocab={len(dictionary)}')